In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)

In [13]:
citation

{'doi': 'https://doi.org/10.1038/s42255-019-0152-6',
 'citation': 'Vijay, J., Gauthier, MF., Biswell, R.L. et al. Single-cell analysis of human adipose tissue identifies depot- and disease-specific cell types. Nat Metab 2, 97–109 (2020).',
 'tables': ['https://static-content.springer.com/esm/art%3A10.1038%2Fs42255-019-0152-6/MediaObjects/42255_2019_152_MOESM3_ESM.xlsx'],
 'organ': 'adipose',
 'species': 'homo_sapiens',
 'reference': 'GRCh38'}

In [14]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

sheet_name = "Supplementary Table 2"
table_name = "clean_degs.xlsx" # local name
ct_map_sheeet_name = "Supplementary Table 3"

## Read in file

In [15]:
col_rename = {
    "P value": "p_raw",
    "Average LogFC": "log_fc",
    "Percentage of cells in cluster expressing the gene": "nnz_frac",
    "Percentage of cells in all other clusters expressing the gene": "nnz_frac_out",
    "Adj. P value": "p_corr",
    "Cluster#": "celltype_num",
    "Gene": "target",
    "Cluster name": "group"
}

col_rename_ct = {
    "Cluster": "group",
    "Manual Annotation": "group_name",
    "Unsupervised Annotation1 (a)": "group_name_detailed"
}

In [16]:
name_map = pd.read_excel(table_name, sheet_name=ct_map_sheeet_name, skiprows=2)#.rename(columns=col_rename_ct).dropna()


In [17]:
expanded_df = (
    name_map.assign(cluster=name_map['Clusters'].str.split(','))  # Split the comma-separated strings
    .explode('cluster')  # Expand the lists into individual rows
    .reset_index(drop=True)  # Reset the index
)

In [18]:
expanded_df.drop_duplicates(subset=["Annotation"]).drop(columns=["Clusters", "Reference", "Gene"]).set_index("cluster")

expanded_df

,Clusters,Annotation,Gene,Reference,cluster
0,"E1,E2",Endothelial cells,"GNG11,SEPW1, SPARCL1, IFI27","Kilari et al., 2013, Ria et al., 2009, Naschbe...",E1
1,"E1,E2",Endothelial cells,"GNG11,SEPW1, SPARCL1, IFI27","Kilari et al., 2013, Ria et al., 2009, Naschbe...",E2
2,E1,Microvascular endothelial cells of adipose tissue,"FABP4, LGALS1, RBP7, GPX3, CD36","Briot et al., 2018",E1
3,E3,Lymphatic endothelial cells,LYVE1,"Banerji et al., 1999",E3
4,IS1,Naive T cells,IL7R,"Schluns et al., 2000, Kondrack et al., 2003",IS1
5,IS6,Activated T cells,"CCL5, IL32","Schall et al., 1988",IS6
6,IS2,Macrophage,CD68,"Chavez-Galan et al., 2015",IS2
7,IS2,Adipose tissue macrophages involved in lipid m...,"LIPA , LPL, CD36, FABP4","Zhang et al., 2017, Grijalva et al., 2016, Aou...",IS2
8,IS9,Macrophage polarization,"FOLR2, KLF4","Jager et al., 2014, Liao et al., 2011",IS9
9,IS10,CD16- monocytes,"S100A8, S100A9, S100A12","Ancuta et al., 2009",IS10


In [19]:
df = pd.read_excel(table_name, sheet_name=sheet_name, skiprows=2).rename(columns=col_rename)
name_map = pd.read_excel(table_name, sheet_name=ct_map_sheeet_name, skiprows=2).rename(columns=col_rename_ct).dropna()

#name_map = name_map[["group", "group_name", "group_name_detailed"]].drop_duplicates().set_index("group")
#df['group'] = df['group'].map(name_map["Annotation"])

#name_map["Clusters"] = name_map["Clusters"].tolist()
#name_map.explode("Clusters")
#name_map = name_map.explode("Clusters")

#name_map
#name_mapping = dict(zip(name_map["Clusters"], name_map["Annotation"])) # convert it into a dictionary

#df["group"] = df["group"].map(name_mapping).fillna(df["group"])

#name_mapping


#df['group'] = name_map['annotation']

df

,p_raw,log_fc,nnz_frac,nnz_frac_out,p_corr,celltype_num,target,group
0,0.000000e+00,2.365577,0.824,0.088,0.000000,0,KRT18,P1
1,0.000000e+00,2.050661,0.674,0.059,0.000000,0,KRT8,P1
2,0.000000e+00,1.911774,0.833,0.111,0.000000,0,ITLN1,P1
3,0.000000e+00,1.718079,0.774,0.123,0.000000,0,TM4SF1,P1
4,0.000000e+00,1.713866,0.542,0.052,0.000000,0,KRT19,P1
...,...,...,...,...,...,...,...,...
1315,1.554997e-07,0.842285,0.435,0.218,0.002847,16,CLU,E3
1316,1.588791e-07,0.495111,0.717,0.452,0.002908,16,GAPDH,E3
1317,1.887556e-07,0.633629,0.446,0.218,0.003455,16,PRDX1,E3
1318,1.974545e-07,0.718422,0.283,0.112,0.003615,16,SERBP1,E3


In [20]:
df.head()

,p_raw,log_fc,nnz_frac,nnz_frac_out,p_corr,celltype_num,target,group
0,0.0,2.365577,0.824,0.088,0.0,0,KRT18,P1
1,0.0,2.050661,0.674,0.059,0.0,0,KRT8,P1
2,0.0,1.911774,0.833,0.111,0.0,0,ITLN1,P1
3,0.0,1.718079,0.774,0.123,0.0,0,TM4SF1,P1
4,0.0,1.713866,0.542,0.052,0.0,0,KRT19,P1


## Unfiltered

In [21]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"group": "cell_type_label", "target": "gene"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["gene_id"] = None

unfiltered_df.drop(['p_raw', 'log_fc', 'nnz_frac_out', 'nnz_frac','p_corr', 'celltype_num'], axis=1, inplace=True) 
save = unfiltered_df.to_dict(orient = "records")
save

[{'gene': 'KRT18',
  'cell_type_label': 'P1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'KRT8',
  'cell_type_label': 'P1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'ITLN1',
  'cell_type_label': 'P1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'TM4SF1',
  'cell_type_label': 'P1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'KRT19',
  'cell_type_label': 'P1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'MSLN',
  'cell_type_label': 'P1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'TFPI2',
  'cell_type_label': 'P1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,

## Save unfiltered

In [22]:
with open('evidence_unfiltered.json', 'w') as f:
    json.dump(save, f, indent = 4)

## Perform the filtering

In [51]:
col_pval = "p_corr"
col_lfc = "log_fc"
col_gene = "target"
col_ct = "group"
col_rank = "nnz_frac"

In [52]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [53]:
markers[col_ct].value_counts()

group
I3    8
I5    7
I6    6
I2    6
E3    5
I1    5
I7    4
P4    4
E2    3
P5    2
E1    1
P1    1
P2    1
Name: count, dtype: int64

In [18]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

group
I7    0.399500
E2    0.503333
I2    0.510833
I6    0.519333
P5    0.549000
P4    0.621500
E1    0.625000
I1    0.637200
E3    0.643400
I3    0.774125
P1    0.824000
P2    0.834000
I5    0.846143
Name: nnz_frac, dtype: float64

## Convert to evidence json

In [56]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = None
save = df.to_dict(orient = "records")

## Save evidence

In [57]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 